# Two-Stage Reranking RAG
## Retrieve Many, Surface the Best — Cross-Encoder Reranking
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/54-reranking-rag/reranking_workbook.ipynb)

Standard RAG uses a **bi-encoder** to retrieve the top-k chunks — fast, but limited in precision. **Two-stage retrieval** adds a second pass: retrieve a large candidate pool cheaply with the bi-encoder, then re-score every candidate with a much more accurate **cross-encoder** and keep only the best. By the end of this workshop you will understand *why* bi-encoders miss things, *how* cross-encoders fix that, and *how* to wire both stages into a LangGraph pipeline.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Bi-encoders vs cross-encoders |
| 2 | **Stage 1** — Vector retrieval: recall over the full corpus |
| 3 | **Stage 2** — FlashRank cross-encoder reranking |
| 4 | **Comparison** — Top-20 raw vs top-4 reranked |
| 5 | **Full Pipeline** — retrieve → rerank → generate with LangGraph |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with `.venv` and `requirements.txt` installed, **or** Google Colab (install cell below)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)
- FlashRank downloads a ~100 MB cross-encoder model on first run (cached after that)

### Key References
> Nogueira, R., & Cho, K. (2019). *Passage Re-ranking with BERT.* https://arxiv.org/abs/1901.04085  
> Reimers, N., & Gurevych, I. (2019). *Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks.* https://arxiv.org/abs/1908.10084  
> Pradeep, R., et al. (2023). *RankLLM: Reranking with LLMs.* https://arxiv.org/abs/2309.15088  
> FlashRank: https://github.com/PrithivirajDamodaran/FlashRank

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "langchain",
        "langchain-openai",
        "langchain-chroma",
        "chromadb",
        "flashrank",
        "langgraph",
        "python-dotenv",
    ])
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
key_ok = bool(key) and key.startswith("sk-")
print(f"OPENAI_API_KEY present and valid: {key_ok}")
if not key_ok:
    print()
    print("  ACTION REQUIRED — add your key before running LLM cells.")
    print("  Local: echo 'OPENAI_API_KEY=sk-...' >> .env")
    print("  Colab: Secrets panel → Add secret 'OPENAI_API_KEY'")

---

## Part 1 — Bi-Encoders vs Cross-Encoders

---

### The Problem with Single-Stage Retrieval

A **bi-encoder** (the model behind ChromaDB, Qdrant, etc.) encodes the query and each document *independently* into vectors, then computes cosine similarity:

```
Query  ──► Encoder ──► q_vec  ─┐
                                ├──► cosine_sim ──► score
Doc    ──► Encoder ──► d_vec  ─┘
```

**Speed advantage:** Documents are encoded offline once and stored in the index. At query time, only the query is encoded — then it's a fast vector search.

**Precision limitation:** Query and document never interact during encoding. The model can't reason about subtle relevance signals (e.g., a document that *contradicts* the query may receive a high similarity score because it shares vocabulary).

---

### How a Cross-Encoder Fixes This

A **cross-encoder** takes the query and document *together* as a single input and outputs a relevance score:

```
[CLS] query [SEP] document [SEP]
              │
         Transformer
              │
        relevance score  (0.0 → 1.0)
```

Every token in the query can attend to every token in the document. This joint attention produces far more accurate relevance judgments.

**Speed limitation:** You can't pre-encode documents — the model must run once per (query, document) pair at query time. Scoring 1 million documents would be prohibitively slow.

---

### Two-Stage Retrieval: Best of Both

```
Full Corpus (N docs)
       │
  Stage 1: Bi-Encoder (fast)
  ─────────────────────────────
  Retrieve top-k candidates          k = 20–100
       │
  Stage 2: Cross-Encoder (precise)
  ─────────────────────────────────
  Re-score all k candidates          k → 3–5
  Keep top-j                         j = 3–5
       │
  LLM Generation
```

| | Bi-Encoder | Cross-Encoder |
|--|-----------|---------------|
| Speed | O(1) query — index lookup | O(k) per query |
| Accuracy | Moderate | High |
| Can pre-index? | Yes | No |
| Typical use | Stage 1: candidate recall | Stage 2: precision scoring |

In [ ]:
# Corpus used throughout the workshop — 20 short technical sentences.
# Dense enough that single-stage retrieval can make mistakes.

SAMPLE_DOCS = [
    "The transformer architecture uses self-attention to process all tokens in parallel.",
    "BERT is a bidirectional encoder trained on masked language modeling and next sentence prediction.",
    "GPT models are autoregressive decoders that generate text left to right, one token at a time.",
    "Self-attention computes a weighted sum of all token representations in a sequence.",
    "Fine-tuning adapts a pretrained language model to a specific downstream task.",
    "Chain-of-thought prompting improves multi-step reasoning by showing intermediate steps.",
    "Retrieval-augmented generation fetches external documents to ground LLM responses.",
    "Vector databases store dense embeddings and support approximate nearest neighbor queries.",
    "Chunking splits documents into smaller pieces to fit within the retrieval context window.",
    "Cross-encoders score query-document pairs jointly, providing higher accuracy than bi-encoders.",
    "Bi-encoders embed query and document independently, enabling fast offline indexing.",
    "Reranking is a two-stage approach: retrieve many candidates cheaply, then score precisely.",
    "Reciprocal Rank Fusion merges multiple ranked lists without requiring calibrated scores.",
    "BM25 is a keyword-based ranking function that works well for exact-match queries.",
    "Hybrid search combines sparse BM25 retrieval with dense vector retrieval for better coverage.",
    "Cosine similarity measures the angle between two embedding vectors, ignoring magnitude.",
    "In-context learning allows LLMs to adapt to new tasks without gradient updates.",
    "Instruction tuning trains models on diverse task descriptions to improve zero-shot performance.",
    "Hallucination occurs when a model generates plausible-sounding but factually incorrect content.",
    "Faithfulness measures whether an answer is supported by the retrieved context.",
]

print(f"Corpus: {len(SAMPLE_DOCS)} documents")
print(f"Avg doc length: {sum(len(d) for d in SAMPLE_DOCS) // len(SAMPLE_DOCS)} chars")
print()
print("First 3 documents:")
for i, d in enumerate(SAMPLE_DOCS[:3]):
    print(f"  [{i:02d}] {d}")

---

## Part 2 — Stage 1: Vector Retrieval

---

### Building the Bi-Encoder Index

We embed all 20 documents with `text-embedding-3-small` and store them in Chroma. At query time, the query is embedded once, and Chroma returns the top-k most similar documents by cosine distance.

```
OFFLINE (index time):
  for each doc → embed(doc) → store(vector, doc)

ONLINE (query time):
  query → embed(query) → cosine_search(top-k) → [doc₁, doc₂, ..., docₖ]
```

**Why retrieve 20 and not just 4?** Because the bi-encoder optimizes for *recall* — it shouldn't miss the relevant document. Returning 20 candidates costs almost nothing extra (all vectors are already in memory), and gives the cross-encoder a larger pool to find the genuinely relevant documents in.

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Stage 1 casts a wide net; Stage 2 precision-cuts. 20:4 is a common ratio —
# going wider (e.g. 50:4) improves recall but costs more cross-encoder compute.
MAX_RETRIEVE = 20   # Stage 1: retrieve this many candidates
MAX_RERANK   = 4    # Stage 2: keep this many after reranking

# text-embedding-3-small: fast and cheap for bi-encoder retrieval.
# Swap to text-embedding-3-large if you need higher semantic fidelity at stage 1.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_texts(
    SAMPLE_DOCS,
    embeddings,
    collection_name="reranking-demo",
)
print(f"Index built: {len(SAMPLE_DOCS)} documents embedded and stored in Chroma.")

In [ ]:
# ─── Stage 1: retrieve top-MAX_RETRIEVE candidates for each query ─────────────

QUERIES = [
    "How do transformers process sequences efficiently?",
    "What is the difference between BERT and GPT architectures?",
    "Why is reranking more accurate than single-stage retrieval?",
]

retrieved: dict[str, list[tuple[str, float]]] = {}  # query → [(doc_text, score)]

# k=MAX_RETRIEVE controls recall: larger k = more chances to capture the right doc,
# but the cross-encoder must score all k candidates — keep k reasonable.
for query in QUERIES:
    results = vectorstore.similarity_search_with_score(query, k=MAX_RETRIEVE)
# Chroma returns L2 distance; 1-score converts to pseudo-cosine-similarity for display.
    retrieved[query] = [(doc.page_content, 1 - score) for doc, score in results]  # dist → sim

# Show top-5 for the first query
q = QUERIES[0]
print(f"Query: '{q}'")
print(f"Top {MAX_RETRIEVE} retrieved (by cosine similarity):")
print()
for rank, (text, sim) in enumerate(retrieved[q][:5], 1):
    print(f"  {rank}. [{sim:.3f}] {text}")

In [ ]:
# ─── Show the full top-20 ranking for query 2 ────────────────────────────────
# This reveals where the bi-encoder makes mistakes — relevant docs buried low.

q2 = QUERIES[1]  # "What is the difference between BERT and GPT?"
print(f"Query: '{q2}'")
print(f"{'Rank':>4}  {'Sim':>6}  Document")
print("-" * 80)
for rank, (text, sim) in enumerate(retrieved[q2], 1):
    # Highlight the two obviously relevant docs
    flag = " ◄ RELEVANT" if "BERT" in text or "GPT" in text else ""
    print(f"  {rank:2d}.  {sim:.3f}  {text[:65]}{flag}")

---

## Part 3 — Stage 2: FlashRank Cross-Encoder Reranking

---

### What is FlashRank?

[FlashRank](https://github.com/PrithivirajDamodaran/FlashRank) is a lightweight Python library that wraps MS-MARCO-trained cross-encoder models. It:

- Runs **fully locally** — no API key required, no network call at inference time
- Downloads the model (~100 MB) automatically on first use and caches it
- Scores a list of `{id, text}` passages against a query and returns them sorted by relevance
- Defaults to `ms-marco-MiniLM-L-12-v2` — a fast, high-quality cross-encoder

### FlashRank API

```python
from flashrank import Ranker, RerankRequest

ranker = Ranker()   # loads model on first call

request = RerankRequest(
    query="your question",
    passages=[{"id": 0, "text": "..."}, {"id": 1, "text": "..."}]
)

results = ranker.rerank(request)
# results is a list of {id, text, score} sorted by score descending
```

**The score is not calibrated** — use it for *ranking*, not as a probability.

In [ ]:
from flashrank import Ranker, RerankRequest

# First call downloads the model (~100 MB). Subsequent calls use the cache.
# Cross-encoders score query+document together (not independently), which is why they
# are far more accurate than bi-encoders — but too slow to rank thousands of docs.
print("Loading FlashRank cross-encoder model...")
ranker = Ranker()
print("FlashRank model ready.")

In [ ]:
# ─── Stage 2: rerank the top-20 candidates for each query ────────────────────

reranked: dict[str, list[dict]] = {}  # query → [{id, text, score}]

for query, candidates in retrieved.items():
    passages = [{"id": i, "text": text} for i, (text, _) in enumerate(candidates)]
    request = RerankRequest(query=query, passages=passages)
    results = ranker.rerank(request)
# FlashRank returns results sorted by score descending — slicing to MAX_RERANK
# is all that's needed to implement the precision cut.
    reranked[query] = results[:MAX_RERANK]  # keep top-4

# Show reranked top-4 for the first query
q = QUERIES[0]
print(f"Query: '{q}'")
print(f"Reranked top-{MAX_RERANK} (FlashRank cross-encoder scores):")
print()
for rank, result in enumerate(reranked[q], 1):
    print(f"  {rank}. [score={result['score']:.4f}] {result['text']}")

---

## Part 4 — Comparison: Top-20 Raw vs Reranked Top-4

---

The key question: did the cross-encoder surface documents that the bi-encoder ranked too low?

We compare:
- **Bi-encoder rank** — position in the top-20 list (1 = most similar)
- **Cross-encoder rank** — position after reranking (1 = most relevant)

Documents that *rose significantly* in rank are cases where the cross-encoder found relevance the bi-encoder missed.

In [ ]:
# ─── Rank comparison for all three queries ────────────────────────────────────

for query in QUERIES:
    candidates = retrieved[query]          # [(text, bi_sim), ...] in rank order
    top4_reranked = reranked[query]        # [{id, text, score}, ...]

    # Map text → bi-encoder rank
    bi_rank = {text: rank for rank, (text, _) in enumerate(candidates, 1)}

    print(f"Query: '{query}'")
    print(f"{'CE rank':>7}  {'BE rank':>7}  {'Δ rank':>7}  Document")
    print("-" * 80)
    for ce_rank, result in enumerate(top4_reranked, 1):
        text = result["text"]
        be = bi_rank.get(text, "?")
        delta = (be - ce_rank) if isinstance(be, int) else "?"
        direction = f"(+{delta} rise)" if isinstance(delta, int) and delta > 0 else ""
        print(f"  {ce_rank:>5}    {str(be):>7}  {str(delta):>7}  {text[:60]}  {direction}")
    print()

In [ ]:
# ─── Cross-encoder score distribution across the full top-20 ─────────────────
# A large drop between the 4th and 5th scores is a good sign:
# it means the top-4 are meaningfully more relevant than the rest.

q3 = QUERIES[2]  # "Why is reranking more accurate?"
candidates = retrieved[q3]
passages = [{"id": i, "text": text} for i, (text, _) in enumerate(candidates)]
all_scores_result = ranker.rerank(RerankRequest(query=q3, passages=passages))

print(f"Query: '{q3}'")
print(f"Full cross-encoder score distribution (all 20 candidates):")
print()
print(f"{'CE rank':>7}  {'Score':>8}  Document")
print("-" * 80)
for i, r in enumerate(all_scores_result, 1):
    keep = " ◄ KEPT" if i <= MAX_RERANK else ""
    separator = "  ─── cut ───" if i == MAX_RERANK else ""
    print(f"  {i:>5}    {r['score']:>8.4f}  {r['text'][:65]}{keep}")
    if separator:
        print(separator)

---

## Part 5 — Full Pipeline: Retrieve → Rerank → Generate

---

We wire both stages into a LangGraph `StateGraph` with three nodes:

```
START
  │
  ▼
retrieve    ─── Chroma: top-20 by cosine similarity
  │
  ▼
rerank      ─── FlashRank: score all 20, keep top-4
  │
  ▼
generate    ─── GPT-4o-mini: answer from top-4 context
  │
  ▼
 END
```

### State Schema

```python
class RerankState(TypedDict):
    query:     str          # input question
    documents: list[str]   # stage-1 candidates
    reranked:  list[str]   # stage-2 top-k
    answer:    str          # final generated answer
```

In [ ]:
from typing import TypedDict

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph


# TypedDict gives LangGraph typed access to state keys; each node returns a dict
# with only the keys it updates — LangGraph merges it into the shared state.
class RerankState(TypedDict):
    query:     str
    documents: list
    reranked:  list
    answer:    str


llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

# temperature=0: deterministic answers for factual RAG — set higher for summaries
# or creative tasks where variation is acceptable.

def retrieve_node(state: RerankState) -> dict:
    docs = vectorstore.similarity_search(state["query"], k=MAX_RETRIEVE)
    return {"documents": [d.page_content for d in docs]}


def rerank_node(state: RerankState) -> dict:
    passages = [{"id": i, "text": t} for i, t in enumerate(state["documents"])]
    request = RerankRequest(query=state["query"], passages=passages)
    results = ranker.rerank(request)
    return {"reranked": [r["text"] for r in results[:MAX_RERANK]]}


def generate_node(state: RerankState) -> dict:
    context = "\n".join(state["reranked"])
    msg = HumanMessage(
        content=f"Answer using ONLY the context below.\n\nContext:\n{context}\n\nQuestion: {state['query']}"
    )
    return {"answer": llm.invoke([msg]).content}


graph = StateGraph(RerankState)
graph.add_node("retrieve", retrieve_node)
graph.add_node("rerank", rerank_node)
graph.add_node("generate", generate_node)
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "rerank")
graph.add_edge("rerank", "generate")
graph.add_edge("generate", END)
# compile() locks the graph topology — no more add_node/add_edge after this.
# Linear graph here: no conditional edges needed since every query goes retrieve→rerank→generate.
app = graph.compile()

print("Pipeline compiled: retrieve → rerank → generate")

In [ ]:
# ─── Run all three queries through the full pipeline ─────────────────────────

for query in QUERIES:
    print(f"{'=' * 65}")
    print(f"Query: {query}")
    result: RerankState = app.invoke(
        {"query": query, "documents": [], "reranked": [], "answer": ""}
    )
    print(f"Retrieved : {len(result['documents'])} candidates")
    print(f"Reranked  : {len(result['reranked'])} kept")
    if result["reranked"]:
        print(f"Top doc   : {result['reranked'][0][:80]}...")
    print(f"Answer    : {result['answer']}")
    print()

---

### Exercise 1 — Change MAX_RETRIEVE and Observe Recall

**Goal:** Run the third query (`"Why is reranking more accurate than single-stage retrieval?"`) with `MAX_RETRIEVE` set to **5**, **10**, and **20**. For each value, check whether the most relevant document (the one about cross-encoders scoring query-document pairs jointly) appears in the final top-4.

**Question:** At what `MAX_RETRIEVE` value does the pipeline first include the most relevant document in the reranked top-4?

In [ ]:
# ── Exercise 1 Starter ─────────────────────────────────────────────────────────

TARGET_DOC = "Cross-encoders score query-document pairs jointly"
QUERY = QUERIES[2]

for k in [5, 10, 20]:
    # TODO: retrieve top-k docs, rerank to top-4, check if TARGET_DOC is in the top-4
    pass

### Exercise 2 — Add a New Document and Observe Rank Changes

**Goal:** Add the following document to the corpus and rebuild the index. Then run the first query (`"How do transformers process sequences efficiently?"`) and compare the reranked top-4 before and after:

```
"Multi-head attention splits the embedding into multiple heads, "
"each learning different types of token relationships in parallel."
```

**Question:** Did this document enter the top-4? At what cross-encoder rank?

In [ ]:
# ── Exercise 2 Starter ─────────────────────────────────────────────────────────

NEW_DOC = (
    "Multi-head attention splits the embedding into multiple heads, "
    "each learning different types of token relationships in parallel."
)

# TODO: Create a new Chroma collection with SAMPLE_DOCS + NEW_DOC
# TODO: Retrieve top-20 for QUERIES[0], rerank, print the top-4
# TODO: Compare to the original top-4 from Part 4
pass

---

## What's Next?

You now understand two-stage retrieval and how FlashRank cross-encoder reranking improves precision over bi-encoder-only search.

### Improve the retrieval stage
- **Example 22 — Hybrid Search RAG** ([`../22-hybrid-search-rag/`](../22-hybrid-search-rag/README.md)): combine BM25 keyword retrieval with vector search before reranking — better recall for exact-match queries
- **Example 53 — Chunking Strategies** ([`../53-chunking-strategies/chunking_workbook.ipynb`](../53-chunking-strategies/chunking_workbook.ipynb)): the chunking strategy determines what goes into your index — reranking can't fix bad chunks

### Improve the reranking stage
- **LangChain `ContextualCompressionRetriever`**: wraps any retriever with a reranker — `FlashrankRerank` is the drop-in compressor
- **Cohere Rerank API**: cloud-based reranker with no local model download (`langchain-cohere`)
- **RankLLM**: use an instruction-tuned LLM itself as the reranker for zero-shot generalization

### Evaluate the improvement
- **Example 16 — RAG Evaluation** ([`../16-rag-eval-notebook/`](../16-rag-eval-notebook/README.md)): score your pipeline on Faithfulness and Context Recall — measure whether reranking actually helped

### Key papers
- Nogueira & Cho (2019). *Passage Re-ranking with BERT.* https://arxiv.org/abs/1901.04085
- Reimers & Gurevych (2019). *Sentence-BERT.* https://arxiv.org/abs/1908.10084
- Pradeep et al. (2023). *RankLLM.* https://arxiv.org/abs/2309.15088

---

*Workshop complete. Next step: Example 22 — add BM25 to the retrieval stage.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself.

### Exercise 1 — Change MAX_RETRIEVE and Observe Recall

**Expected behaviour:**

| MAX_RETRIEVE | Target doc in top-4? | Why |
|-------------|---------------------|-----|
| 5 | May miss it | Only 5 candidates for the cross-encoder to work with — if bi-encoder ranks the target #6+, it's gone |
| 10 | Likely present | Sufficient candidate pool for FlashRank to find it |
| 20 | Yes | Maximum recall — cross-encoder has the most to work with |

**Key insight:** The bi-encoder controls recall. If the relevant document isn't in the top-k candidate pool, the cross-encoder can never promote it. Always set `MAX_RETRIEVE` large enough that the target recall is high (commonly 20–100).

In [ ]:
# Exercise 1 — sample solution

TARGET_DOC = "Cross-encoders score query-document pairs jointly"
QUERY = QUERIES[2]

print(f"Query: '{QUERY}'")
print(f"Target: '{TARGET_DOC[:50]}...'")
print()
print(f"{'k':>4}  {'In top-4?':>10}  {'BE rank of target':>18}")
print("-" * 38)

for k in [5, 10, 20]:
    docs = vectorstore.similarity_search(QUERY, k=k)
    texts = [d.page_content for d in docs]

    # bi-encoder rank of the target
    be_rank = next((i + 1 for i, t in enumerate(texts) if TARGET_DOC in t), "not in pool")

    # rerank and check top-4
    passages = [{"id": i, "text": t} for i, t in enumerate(texts)]
    results = ranker.rerank(RerankRequest(query=QUERY, passages=passages))
    top4 = [r["text"] for r in results[:4]]
    found = any(TARGET_DOC in t for t in top4)

    print(f"  {k:>2}  {'YES' if found else 'NO':>10}  {str(be_rank):>18}")

### Exercise 2 — Add a New Document and Observe Rank Changes

**Expected behaviour:** The new multi-head attention document is semantically very close to the query about transformer efficiency. The cross-encoder should score it highly because it directly describes a mechanism that makes transformers efficient. It will likely enter the top-2 in the reranked list, displacing a less directly relevant document.

In [ ]:
# Exercise 2 — sample solution

NEW_DOC = (
    "Multi-head attention splits the embedding into multiple heads, "
    "each learning different types of token relationships in parallel."
)

extended_docs = SAMPLE_DOCS + [NEW_DOC]
vs_extended = Chroma.from_texts(
    extended_docs, embeddings, collection_name="reranking-extended"
)

q = QUERIES[0]
docs_ext = vs_extended.similarity_search(q, k=MAX_RETRIEVE)
passages_ext = [{"id": i, "text": d.page_content} for i, d in enumerate(docs_ext)]
results_ext = ranker.rerank(RerankRequest(query=q, passages=passages_ext))
top4_ext = results_ext[:MAX_RERANK]

print(f"Query: '{q}'")
print()
print("Reranked top-4 WITH new document:")
for rank, r in enumerate(top4_ext, 1):
    flag = " ◄ NEW" if NEW_DOC in r["text"] else ""
    print(f"  {rank}. [score={r['score']:.4f}] {r['text'][:70]}{flag}")

print()
print("Original reranked top-4 (without new document):")
for rank, r in enumerate(reranked[q], 1):
    print(f"  {rank}. [score={r['score']:.4f}] {r['text'][:70]}")